In [1]:
pip install ESM

  Using cached esm-3.2.1.post1-py3-none-any.whl.metadata (18 kB)
  Using cached torchtext-0.6.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached transformers-4.48.1-py3-none-any.whl.metadata (44 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached msgpack_numpy-0.4.8-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 1.1 MB/s  0:00:03 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) .

In [21]:
import os
import numpy as np
import pandas as pd

class PDBparser:
    
    def __init__(self, pdbPath):
        self.__pdbPath = pdbPath

    def get_residues(self):
        with open(self.__pdbPath, 'r') as file:
            aa_map = {
                'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
                'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G',
                'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
                'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
                'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}
            
            seen_residues = {}

            for line in file:
                line = line.strip()

                if line.startswith('ATOM'):
                    res_name = line[17:20]
                    chain = line[21]
                    res_num = int(line[22:26])

                    if chain not in seen_residues:
                        seen_residues[chain] = {}

                    if res_num not in seen_residues[chain]:
                        seen_residues[chain][res_num] = aa_map.get(res_name, 'err')
            
            sequences = {}
            for chain, residues in seen_residues.items():
                sequences[chain] = "".join(residues[num] for num in sorted(residues))     

        df = pd.DataFrame(
            list(sequences.items()),
            columns=['chain', 'sequence']
        )

        return df


pdb = PDBparser("/Users/austingilbride/masters_sem_2/SBI/Predicting_Binding_Sites/1CY1.pdb")
pdb_parsed = pdb.get_residues()
pdb_parsed.head()

,chain,sequence
0,A,GKALVIVESPAKAKTINKYLGSDYVVKSSVGHIRDLPRGALVNRMG...


In [ ]:
'''#Input should be sequence in string format, output should be df with pdb_id, chain_id, residue, and n_esm embeddings columns
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
import torch

class ESM_df:
    """Apply ESM model to get protein embeddings in df format from a PDB sequence
    Methods:
        get_embeddings == gets us the embeddings from the ESM of a PDB file 
    """

    def __init__(self, pdb_object):
        self.pdb_object = pdb_object
        
    def get_embeddings(self, pdb_object):
        client = ESMC.from_pretrained("esmc_300m")
        os.makedirs("embeddings_proteins2", exist_ok=True)
        dictionary = {}
        for _, row in self.pdb_object.iterrows():
            chain = row["chain"]
            sequence_full = str(row["sequence"])

            protein = ESMProtein(sequence=sequence_full)
            protein_tensor = client.encode(protein)
            
            with torch.no_grad():
                output = client.logits(
                protein_tensor,
                LogitsConfig(sequence=True, return_embeddings=True)
                )      
            embeddings = output.embeddings.squeeze(0).half().cpu().numpy()[1:-1, :]
            dictionary[chain] = embeddings
        

        
        output = pd.DataFrame.from_dict(dictionary, orient="index", columns=["embeddings"])
        output = output.reset_index().rename(columns={'index': "chain"})
        return output
'''

In [ ]:
import os
import pandas as pd
import torch
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

class ESM_df:

    def __init__(self, pdb_df, pdb_id=None):
        self.pdb_df = pdb_df
        self.pdb_id = pdb_id

    def get_embeddings(self):
        client = ESMC.from_pretrained("esmc_300m")
        os.makedirs("embeddings_proteins2", exist_ok=True)

        all_rows = []

        for _, row in self.pdb_df.iterrows():
            chain_id = row["chain"]
            sequence_full = str(row["sequence"])
            residue_list = list(sequence_full)

            protein = ESMProtein(sequence=sequence_full)
            protein_tensor = client.encode(protein)

            with torch.no_grad():
                output = client.logits(
                    protein_tensor,
                    LogitsConfig(sequence=True, return_embeddings=True)
                )

            
            embeddings = output.embeddings.squeeze(0).cpu().numpy()[1:-1, :]

            # safety check from chat, developed while debugging. 
            if embeddings.shape[0] != len(residue_list):
                raise ValueError(
                    f"Mismatch for chain {chain_id}: "
                    f"{embeddings.shape[0]} embeddings vs {len(residue_list)} residues"
                )

            emb_cols = [f"emb_{i}" for i in range(embeddings.shape[1])]
            chain_df = pd.DataFrame(embeddings, columns=emb_cols)

            chain_df.insert(0, "residue", residue_list)
            chain_df.insert(0, "chain_id", chain_id)
            chain_df.insert(0, "pdb_id", self.pdb_id)

            all_rows.append(chain_df)

        final_df = pd.concat(all_rows, ignore_index=True)
        return final_df

In [26]:
embeddings = ESM_df(pdb_parsed, "1CY1")
df_with_embeddings = embeddings.get_embeddings()

df_with_embeddings.head()

,pdb_id,chain_id,residue,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,emb_6,...,emb_950,emb_951,emb_952,emb_953,emb_954,emb_955,emb_956,emb_957,emb_958,emb_959
0,1CY1,A,G,-0.006309,-0.040782,0.018655,-0.010588,0.008218,0.037279,-0.004148,...,0.030521,0.002574,0.026207,0.048397,0.024329,0.004088,-0.012002,0.034533,0.043613,0.060083
1,1CY1,A,K,0.000097,0.015087,-0.025818,-0.028518,-0.041529,0.019195,-0.004833,...,0.012599,-0.026920,-0.054776,0.057457,-0.003940,0.043067,-0.021521,0.061916,0.035811,0.018635
2,1CY1,A,A,0.048292,0.014519,-0.041627,-0.027983,0.010406,0.026925,-0.017442,...,-0.036829,-0.005102,0.042052,0.001421,0.010493,-0.043738,-0.031952,0.007599,0.002892,-0.013188
3,1CY1,A,L,-0.001959,-0.111996,0.022011,-0.004863,0.011412,0.017976,-0.026455,...,-0.053971,-0.034352,-0.018874,-0.041282,0.020682,-0.011068,0.071930,0.016299,0.001028,-0.036806
4,1CY1,A,V,0.019288,-0.033036,-0.023495,-0.070523,-0.027506,-0.006572,-0.025732,...,0.025228,-0.015312,-0.002438,-0.031381,-0.036912,-0.022695,0.015744,-0.025004,0.025090,-0.051024
